# Contextual sarcasm detection in news headlines

Training notebook for Helal et al. (2024), *A contextual-based approach for sarcasm detection*.

**What this notebook does**
1. Loads the News Headlines dataset
2. Builds RoBERTa inputs (`HEADLINE: ...`)
3. Fine-tunes **RoBERTa-base**
4. Saves a model folder you can reuse later for inference or an app

**Saved model location**

`../models/roberta_headline_only/`

That folder is the "model file". Keep the whole folder (`config.json`, tokenizer files, and `model.safetensors` or `pytorch_model.bin`).


## 1. Introduction

Paper setup we follow:
- Model: `roberta-base` → classification head → 2 logits
- Loss: CrossEntropyLoss (used inside `RobertaForSequenceClassification`)
- Optimizer: AdamW
- Paper Table 4: batch 32, lr `5e-5`, 5 epochs, eval batch 64, weight decay 0.01
- Split: 80/20 train/test, then 12.5% of train as validation (about 70/10/20)

This machine may be **CPU-only**. Full training on 26,709 headlines for 5 epochs can take many hours. Use the config cell below to start with a smaller sample, then set `MAX_SAMPLES = None` for the paper-scale run.

Context fields (`author`, `section`, `description`) are still `unknown` until articles are scraped, so the meaningful experiment **right now** is `headline_only`.


## Config — change this before you train


In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "utils.py").exists():
            return candidate
    raise RuntimeError("Could not find the project root (folder that contains src/).")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Paper setting: MAX_SAMPLES = None and EPOCHS = 5
# Faster CPU smoke test: MAX_SAMPLES = 2000 and EPOCHS = 2
MAX_SAMPLES = 2000
EPOCHS = 2
CONTEXT_TYPE = "headline_only"  # later: author, section, description, all_context
MODEL_DIR = PROJECT_ROOT / "models" / "roberta_headline_only"

print("Project root:", PROJECT_ROOT)
print("MAX_SAMPLES:", MAX_SAMPLES)
print("EPOCHS:", EPOCHS)
print("CONTEXT_TYPE:", CONTEXT_TYPE)
print("MODEL_DIR:", MODEL_DIR)


## 2. Dataset loading


In [ ]:
import pandas as pd
from preprocessing import run_inspection
from utils import PROCESSED_CSV, get_device, get_gpu_name, ensure_project_dirs

ensure_project_dirs()
if not PROCESSED_CSV.exists():
    run_inspection()

df = pd.read_csv(PROCESSED_CSV)
print("Processed file:", PROCESSED_CSV)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)


## 3. Dataset inspection


In [ ]:
print("Missing values:")
print(df.isna().sum())
print("\nLabel counts:")
print(df["is_sarcastic"].value_counts())
print("\nDuplicate headlines:", int(df["headline"].duplicated().sum()))
print("Duplicate article links:", int(df["article_link"].duplicated().sum()))
print("\nDevice:", get_device())
print("GPU:", get_gpu_name() or "None (CPU)")


## 4. Class distribution


In [ ]:
import matplotlib.pyplot as plt

counts = df["is_sarcastic"].value_counts().sort_index()
labels = ["Non-sarcastic (0)", "Sarcastic (1)"]
values = [int(counts.get(0, 0)), int(counts.get(1, 0))]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, values, color=["#4C78A8", "#F58518"])
ax.set_ylabel("Count")
ax.set_title("News Headlines class distribution")
for i, value in enumerate(values):
    ax.text(i, value, f"{value:,}", ha="center", va="bottom")
fig.tight_layout()
plt.show()


## 5. Contextual data inspection

The original dataset only has headline + article link + label. Author / section / description were scraped in the paper. Until we scrape, those columns are the paper's missing-value placeholder: `unknown`.

`source_family` is **not** a model feature. It is here to show leakage: Onion ≈ sarcastic, HuffPost ≈ non-sarcastic.


In [ ]:
from dataset import context_fields_are_populated

for col in ["author", "section", "description"]:
    unknown_share = (df[col].fillna("unknown").str.lower() == "unknown").mean()
    print(f"{col}: {unknown_share:.1%} unknown")

print("Real scraped context available:", context_fields_are_populated(df))
print("\nSource vs label:")
print(pd.crosstab(df["source_family"], df["is_sarcastic"]))


## 6. Data preprocessing / input construction

RoBERTa should see natural language. We do not stem, lemmatize, or remove stopwords.

Inputs use a clear field separator, for example:

```
HEADLINE: Scientists discover surprising result
AUTHOR: John Smith
```


In [ ]:
from dataset import add_input_text, build_input_text

example = df.iloc[0]
print(build_input_text(example, "headline_only"))
print("---")
print(build_input_text(example, "all_context"))

work_df = add_input_text(df, CONTEXT_TYPE)
print("\nExample training string:")
print(work_df.loc[0, "input_text"])


## 7. Tokenization


In [ ]:
from dataset import maybe_sample, token_length_stats
from model import load_tokenizer
from utils import MODEL_NAME, MAX_LENGTH

sampled = maybe_sample(work_df, MAX_SAMPLES)
tokenizer = load_tokenizer(MODEL_NAME)
stats = token_length_stats(sampled["input_text"].tolist(), tokenizer)
lengths = stats.pop("lengths")
print("Tokenizer:", MODEL_NAME)
print("Configured MAX_LENGTH:", MAX_LENGTH)
print(stats)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(lengths, bins=30, color="#4C78A8")
ax.axvline(MAX_LENGTH, color="red", linestyle="--", label=f"MAX_LENGTH={MAX_LENGTH}")
ax.set_xlabel("Tokens (no truncation)")
ax.set_ylabel("Headlines")
ax.set_title("Token length distribution")
ax.legend()
fig.tight_layout()
plt.show()


## 8. Model definition

```
headline (+ optional context)
        ↓
RoBERTa tokenizer
        ↓
RoBERTa-base (12 encoder layers)
        ↓
classification head
        ↓
2 logits → softmax → sarcastic / non-sarcastic
```


In [ ]:
from model import load_model
from utils import NUM_LABELS

model = load_model(MODEL_NAME, num_labels=NUM_LABELS)
print(type(model).__name__)
print("num_labels:", model.config.num_labels)
print("hidden layers:", model.config.num_hidden_layers)
del model  # free memory; train.py will load a fresh copy


## 9. Train / validation / test split

Paper: 80% train, 20% test.

Our implementation also holds out 12.5% of that training portion as validation (about **70% train / 10% val / 20% test**), stratified on `is_sarcastic`. Labels never go into the input text.


In [ ]:
from dataset import stratified_splits

split_df = maybe_sample(work_df, MAX_SAMPLES)
train_df, val_df, test_df = stratified_splits(split_df)
print("Dataset size:", len(split_df))
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Train sarcastic rate:", train_df["is_sarcastic"].mean().round(4))
print("Val sarcastic rate:", val_df["is_sarcastic"].mean().round(4))
print("Test sarcastic rate:", test_df["is_sarcastic"].mean().round(4))


## 10. Baseline training (headline only) — this saves the model

This cell is the one that creates the reusable model folder.

After it finishes, copy or zip:

`models/roberta_headline_only/`

Then load that folder in inference or in another app.


In [ ]:
from train import run_experiment

result = run_experiment(
    context_type=CONTEXT_TYPE,
    max_samples=MAX_SAMPLES,
    model_dir=MODEL_DIR,
    epochs=EPOCHS,
)

result["model_dir"]


## 11. Context-model training

Train `all_context` only after scraping has filled author / section / description. If those fields are still `unknown`, this run would look almost identical to headline-only.

```python
# Uncomment after scraping
# context_result = run_experiment(
#     context_type="all_context",
#     max_samples=MAX_SAMPLES,
#     model_dir=PROJECT_ROOT / "models" / "roberta_all_context",
#     epochs=EPOCHS,
# )
```


## 12. Ablation experiments

Run each setting separately when context data exists. Do not assume which feature helps; compare F1 from our runs, not the paper's 99.7%.

| Experiment | `context_type` |
| --- | --- |
| A Headline only | `headline_only` |
| B + Author | `author` |
| C + Section | `section` |
| D + Description | `description` |
| E All context | `all_context` |


In [ ]:
from dataset import context_fields_are_populated

if not context_fields_are_populated(df):
    print("Skipping ablation loop: author/section/description are still unknown.")
    print("Headline-only results are in result / results/experiment_results.csv.")
else:
    ablation_types = ["headline_only", "author", "section", "description", "all_context"]
    print("Context is populated. Example loop:")
    print(ablation_types)
    # for name in ablation_types:
    #     run_experiment(context_type=name, max_samples=MAX_SAMPLES,
    #                    model_dir=PROJECT_ROOT / "models" / f"roberta_{name}",
    #                    epochs=EPOCHS)


## 13. Evaluation


In [ ]:
print("Test accuracy :", round(result["accuracy"], 4))
print("Test precision:", round(result["precision"], 4))
print("Test recall   :", round(result["recall"], 4))
print("Test F1       :", round(result["f1"], 4))
print("Macro F1      :", round(result["macro_f1"], 4))
print("Weighted F1   :", round(result["weighted_f1"], 4))
print("Train time (s):", result["training_time_sec"])
print("Test time (s) :", result["inference_time_sec"])
print()
print(result["classification_report"])


## 14. Confusion matrix


In [ ]:
from IPython.display import Image, display
display(Image(filename=result["confusion_matrix_path"]))


## 15. Training curves


In [ ]:
display(Image(filename=result["loss_curve_path"]))
pd.DataFrame(result["history"])


## 16. Error analysis

`results/error_analysis.csv` stores true positives, true negatives, false positives, and false negatives with the sarcasm probability.


In [ ]:
error_path = PROJECT_ROOT / "results" / "error_analysis.csv"
errors = pd.read_csv(error_path)
print(errors["error_type"].value_counts())
errors.groupby("error_type").head(2)[
    ["error_type", "headline", "true_label", "predicted_label", "sarcasm_probability"]
]


## 17. Results comparison


In [ ]:
results_csv = PROJECT_ROOT / "results" / "experiment_results.csv"
summary = pd.read_csv(results_csv)
summary[["experiment", "context", "accuracy", "precision", "recall", "f1"]]


## 18. Conclusions and how to reuse the model

**Do not report the paper's 99.7% F1 unless this notebook actually reached it.** Use the metrics printed above.

### Where the model is

The usable model is the **folder**:

`models/roberta_headline_only/`

Copy that whole folder. You need more than one file: weights + `config.json` + tokenizer files.

### Load it later in another notebook or app


In [ ]:
from inference import predict_one
from model import load_saved_classifier

print("Saved model folder:", MODEL_DIR)
print("Exists:", MODEL_DIR.exists())
print("Files:", sorted(p.name for p in MODEL_DIR.glob("*")) if MODEL_DIR.exists() else [])

# Example later usage:
# tokenizer, model = load_saved_classifier(MODEL_DIR)
# predict_one("mom starting to fear son's web series closest thing she will have to grandchild", model_dir=MODEL_DIR)


### Command-line inference after training

From the project root:

```bash
python src/inference.py --model_dir models/roberta_headline_only
```

Then enter Headline / Author / Section / Description. Author, section, and description can be left blank for a headline-only model.

To train the **full** paper-scale run, set `MAX_SAMPLES = None` and `EPOCHS = 5` in the config cell and re-run the training cell. A GPU (Colab T4, as in the paper) is strongly recommended for that.
